# Project 4: Collisions of hard disks

In this notebook, we explore the *hard-disk model*, a classic system in statistical mechanics
consisting of impermeable disks moving freely within a confined space and undergoing elastic
collisions upon contact. With no long-range forces, friction, or external thermal baths,
this model strips physics down to pure geometry and momentum transfer.

Despite its simplicity, it holds an important place in physics history. Pioneered by Ludwig
Boltzmann to lay the foundations of the kinetic theory of gases, the model continues to yield
surprises today, such as its two-dimensional melting transition, which was only fully resolved
in 2011 through advanced computer simulations.

We will simulate it with the algorithm proposed by Alder and Wainwright in 1957, one of the
very first *molecular dynamics* simulations. Instead of moving the disks with small time steps,
it jumps directly from one collision to the next: it is an *event-driven* algorithm.

- **Part 1**: a single disk bouncing in a box, a pinball,
- **Part 2**: watching it move with an animation,
- **Part 3**: the collision between two disks,
- **Part 4**: many disks in a box, the full Alder and Wainwright algorithm,
- **Part 5**: from Newton to Boltzmann, the distribution of the velocities.

All the disks have the same mass, $m = 1$. The cell below defines a few functions to draw and
animate the disks, so that you can focus on the physics and on the algorithm.

In [ ]:
# Common imports
%matplotlib inline
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.collections import EllipseCollection
from IPython.display import HTML

PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#56B4E9", "#E69F00", "#F0E442"]
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (6.5, 4.2),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "axes.edgecolor": "#898781",
    "axes.linewidth": 0.9,
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "legend.frameon": False,
    "lines.linewidth": 2.0,
    "axes.axisbelow": True,
    "animation.html": "jshtml",    # animations are rendered as interactive JavaScript widgets
    "animation.embed_limit": 60,   # in MB
})

# the random number generator
rng = np.random.default_rng(seed=2026)

# colors of the disks and of the walls
DISK_COLOR, EDGE_COLOR, WALL_COLOR = "#56B4E9", "#0072B2", "#52514e"


def get_positions(system) -> np.ndarray:
    """The positions of the disks of a system, as an array of shape (n_disks, 2)."""
    positions = system.positions if hasattr(system, "positions") else system.position
    return np.array(positions, ndmin=2)  # a copy: the drawing must not change when the disks move


def draw_disks(system, ax: plt.Axes | None = None, box: tuple[float, float] | None = (1.0, 1.0),
               limits: tuple[tuple[float, float], tuple[float, float]] | None = None) -> EllipseCollection:
    """Draw the disks of a system (an object with a radius and position(s) attributes).

    The disks are drawn in the (Lx, Ly) box or without walls if box is None.
    The size of the disks is in data units, so that touching disks really touch.
    """
    ax = ax if ax is not None else plt.gca()
    if box is not None:
        ax.add_patch(plt.Rectangle((0, 0), *box, fill=False, ec=WALL_COLOR, lw=2, zorder=3))
        limits = limits or ((0, box[0]), (0, box[1]))
    diameter = 2 * system.radius
    disks = EllipseCollection(diameter, diameter, 0, units="xy", offsets=get_positions(system),
                              offset_transform=ax.transData, facecolors=DISK_COLOR,
                              edgecolors=EDGE_COLOR, linewidths=1.5)
    ax.add_collection(disks)
    ax.set(xlim=limits[0], ylim=limits[1], aspect="equal")
    ax.set_axis_off()
    return disks


def animate_disks(system, dt: float, n_frames: int = 200, **kwargs) -> HTML:
    """Animate a system with a method step(dt), calling step(dt) between two frames.

    The keyword arguments are passed to draw_disks.
    """
    fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=80)
    disks = draw_disks(system, ax, **kwargs)
    plt.close(fig)  # do not show the static figure as well

    def make_frame(frame: int):
        if frame > 0:  # the first frame shows the initial condition
            system.step(dt)
        disks.set_offsets(get_positions(system))
        return [disks]

    ani = animation.FuncAnimation(fig, make_frame, frames=n_frames, interval=30, blit=True)
    return HTML(ani.to_jshtml())


def random_positions(n_disks: int, radius: float, rng: np.random.Generator, box: float = 1.0) -> np.ndarray:
    """Positions of n_disks non-overlapping disks in the box [0, box]^2, placed one by one."""
    positions = []
    while len(positions) < n_disks:
        candidate = rng.uniform(radius, box - radius, size=2)
        if all(np.sum((candidate - p) ** 2) > (2 * radius) ** 2 for p in positions):
            positions.append(candidate)
    return np.array(positions)

## Part 1: A single pinball disk

Let us start with a single disk of radius $r$ in a box $L_x \times L_y$. It flies in a
straight line, $\vec r(t) = \vec r(0) + \vec v\, t$, until it hits a wall. For an elastic
collision with a vertical wall, $v_x \to -v_x$, and with a horizontal wall, $v_y \to -v_y$.

There are four possible *events*: the collisions with the right, top, left and bottom walls.
The *events table* is an array of four numbers, the times until each of these events. For
instance, if the disk moves to the right ($v_x > 0$), it hits the right wall when its center
reaches $x = L_x - r$, after a time $(L_x - r - x) / v_x$. If it moves to the left, it never
hits the right wall: the corresponding event is impossible.

The event-driven algorithm is then:

1. find the smallest time $t_\mathrm{coll}$ in the events table: this is the next event,
2. move the disk during $t_\mathrm{coll}$, to the point of contact with the wall,
3. reverse the right velocity component and update the events table.

**The `nan` trick.** Impossible events can be marked with `np.nan` (*not a number*). The
function `np.nanargmin` finds the index of the smallest element of an array and ignores
the `nan` entries:

```python
events = np.array([2.2, 5.5, np.nan, np.nan])
np.nanargmin(events)   # 0: the right wall comes first
```

Write a class with the following structure:

```python
class SingleDisk:
    """A disk of radius r bouncing in the box Lx x Ly."""

    def __init__(self, radius: float, box: tuple[float, float],
                 position: np.ndarray, velocity: np.ndarray) -> None:
        ...
        self.events = np.full(4, np.nan)  # right, top, left, bottom
        self.update_events()

    def update_events(self) -> None:
        """Compute the times until the disk hits each of the four walls."""
        ...

    def next_event(self) -> float:
        """Move the disk to the next event, change its velocity and return the elapsed time."""
        ...
```

*Hints*: Store copies of the arrays, `self.position = np.array(position, dtype=float)`, otherwise
the object would modify the arrays of the caller. With the convention right, top, left,
bottom for the walls, `index % 2` tells you which velocity component is reversed.

### Check your code

Create a disk of radius $0.3$ at $(2.5, 2.5)$ with velocity $(1, 0.4)$ in a box of size $5
\times 5$. Print its events table: the right wall should come first, after a time $2.2$.
Call `next_event()` a few times and draw the trajectory, using `draw_disks(disk, box=(5, 5))`
to draw the disk. After every event, the disk should touch a wall. These are exactly the
snapshots shown in the lecture.

### Solution

The events table is computed for the four walls at once. Dividing by a velocity component that
is zero would give a warning, which `np.errstate` silences: the corresponding event is
impossible anyway, and `np.where` replaces it by `nan`.

In [ ]:
class SingleDisk:
    """A disk of radius r bouncing in the box Lx x Ly."""

    def __init__(self, radius: float, box: tuple[float, float],
                 position: np.ndarray, velocity: np.ndarray) -> None:
        self.radius = radius
        self.box = np.array(box, dtype=float)
        self.position = np.array(position, dtype=float)
        self.velocity = np.array(velocity, dtype=float)
        self.events = np.full(4, np.nan)  # right, top, left, bottom
        self.update_events()

    def update_events(self) -> None:
        """Compute the times until the disk hits each of the four walls."""
        with np.errstate(divide="ignore", invalid="ignore"):
            to_upper_walls = (self.box - self.radius - self.position) / self.velocity  # right, top
            to_lower_walls = (self.radius - self.position) / self.velocity             # left, bottom
        times = np.concatenate([to_upper_walls, to_lower_walls])
        moving_towards = np.concatenate([self.velocity > 0, self.velocity < 0])
        self.events = np.where(moving_towards, times, np.nan)

    def next_event(self) -> float:
        """Move the disk to the next event, change its velocity and return the elapsed time."""
        wall = np.nanargmin(self.events)
        t_coll = self.events[wall]
        self.position += t_coll * self.velocity
        self.velocity[wall % 2] *= -1  # walls 0, 2 are vertical: v_x; walls 1, 3 are horizontal: v_y
        self.update_events()
        return t_coll

Let's follow the disk during six events and draw a snapshot after each of them.

In [ ]:
disk = SingleDisk(radius=0.3, box=(5, 5), position=(2.5, 2.5), velocity=(1, 0.4))
wall_names = ["right", "top", "left", "bottom"]

fig, axes = plt.subplots(1, 6, figsize=(15, 3), layout="constrained")
trajectory = [disk.position.copy()]
for n, ax in enumerate(axes):
    draw_disks(disk, ax, box=(5, 5))
    ax.plot(*np.array(trajectory).T, ls="dashed", color=WALL_COLOR, lw=1)
    ax.arrow(*disk.position, *disk.velocity, color=PALETTE[1], width=0.05, head_width=0.25, zorder=4)
    table = "  ".join("X" if np.isnan(t) else f"{t:.1f}" for t in disk.events)
    ax.set_title(f"event {n}" if n else "start", fontsize=12)
    ax.text(2.5, -0.3, table, ha="center", va="top", fontsize=11, family="monospace")
    print(f"{n}: position = {disk.position}, velocity = {disk.velocity}, "
          f"next: {wall_names[np.nanargmin(disk.events)]} wall")
    disk.next_event()
    trajectory.append(disk.position.copy())

The tables below each snapshot (right, top, left, bottom) are the ones of the lecture. The disk
always touches a wall after an event, and its trajectory is a nice billiard path.

## Part 2: Let's watch the disk move

It would be nice to see the disk moving in an animation. But every call to `next_event()` makes
the disk jump directly to the next wall: this is the whole point of the event-driven algorithm,
but a movie made this way would only show the disk stuck to the walls! A movie needs frames at
*regular* times, while the events happen at *irregular* times.

Add a method `step(dt)` to your class, which evolves the system during a fixed time `dt`:

```python
def step(self, dt: float) -> None:
    """Evolve the disk during a time dt, processing all the events on the way."""
    while (next event happens before dt):
        dt -= self.next_event()
    ...  # fly for the remaining time dt, and remember that the events are now closer!
```

There can be several events, or none at all, during `dt`. Once all the events before `dt` have
been processed, the disk flies for the remaining time, and all the times in the events table
decrease by this amount.

Then use the function `animate_disks` defined at the top of the notebook to see the disk move:

```python
disk = SingleDisk(radius=0.3, box=(5, 5), position=(2.5, 2.5), velocity=(1, 0.423))
animate_disks(disk, dt=0.1, n_frames=200, box=(5, 5))
```

*Hint*: To add a method to a class that you already wrote, you can define a subclass,
`class MovingDisk(SingleDisk):`, which only contains the new method `step`. It inherits all the
other methods from `SingleDisk`. Or simply go back to Part 1 and add the method there.

*Optional*: Try a box of size $5 \times 3$ and other velocities. When is the trajectory
periodic?

### Solution

We use a subclass, which inherits everything from `SingleDisk`. Note the `while` loop: several
walls can be hit during one step, in a corner for instance.

In [ ]:
class MovingDisk(SingleDisk):
    """A single disk with a method step(dt), to make animations."""

    def step(self, dt: float) -> None:
        """Evolve the disk during a time dt, processing all the events on the way."""
        while np.nanmin(self.events) <= dt:
            dt -= self.next_event()
        self.position += dt * self.velocity
        self.events -= dt


disk = MovingDisk(radius=0.3, box=(5, 5), position=(2.5, 2.5), velocity=(1, 0.423))
animate_disks(disk, dt=0.1, n_frames=200, box=(5, 5))

In a box $L_x \times L_y$, the motion of the center along $x$ and along $y$ are two independent
back-and-forth motions on segments of lengths $L_x - 2r$ and $L_y - 2r$, with periods
$2(L_x - 2r)/|v_x|$ and $2(L_y - 2r)/|v_y|$. The trajectory is periodic when the ratio of the two
periods is a rational number, and otherwise it fills the whole box after a long time. Let's
check this with a long run of $500$ events:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), layout="constrained")
for ax, velocity, title in [(axes[0], (1, 0.5), "$v_y / v_x = 1/2$: periodic"),
                            (axes[1], (1, 0.5 * np.sqrt(2)), r"$v_y / v_x = 1/\sqrt{2}$: fills the box")]:
    disk = MovingDisk(radius=0.3, box=(5, 5), position=(1.5, 1.0), velocity=velocity)
    trajectory = [disk.position.copy()]
    for _ in range(500):
        disk.next_event()
        trajectory.append(disk.position.copy())
    ax.plot(*np.array(trajectory).T, color=PALETTE[0], lw=0.4)
    draw_disks(disk, ax, box=(5, 5))
    ax.set_title(title)

## Part 3: Two-disk collision

Before we put many disks in the box, let us study the collision of two disks of the same radius
$r$, without any wall. With $\Delta\vec r = \vec r_2 - \vec r_1$ and
$\Delta\vec v = \vec v_2 - \vec v_1$, the disks touch when their distance is $2r$:

\begin{equation}
  \big|\Delta\vec r + \Delta\vec v\, t_\mathrm{coll}\big| = 2r
\end{equation}

This is a quadratic equation $a t^2 + 2 b t + c = 0$, with
$a = |\Delta\vec v|^2$, $b = \Delta\vec r \cdot \Delta\vec v$ and $c = |\Delta\vec r|^2 - 4r^2$.
The time of the first contact is the smallest root

\begin{equation}
  t_\mathrm{coll} = \frac{-b - \sqrt{b^2 - ac}}{a}
\end{equation}

and a collision only takes place if the disks approach each other and do not miss each other:

\begin{equation}
  b = \Delta\vec r \cdot \Delta\vec v < 0 \qquad \text{and} \qquad b^2 - ac \ge 0
\end{equation}

At the collision, the disks exchange momentum along the unit vector
$\hat r = \Delta\vec r / |\Delta\vec r|$ that connects their centers. For equal masses,
conservation of momentum and energy then gives (see the lecture)

\begin{equation}
  \vec v'_1 = \vec v_1 - \Delta\vec w \qquad
  \vec v'_2 = \vec v_2 + \Delta\vec w \qquad
  \text{with} \qquad \Delta\vec w = -\left(\Delta\vec v \cdot \hat r\right) \hat r
\end{equation}

In words, the two disks swap the components of their velocities along $\hat r$.

1. Write a function `collision_time(dr, dv, sigma)` that returns the time until two disks at
   relative position `dr` with relative velocity `dv` are at a distance `sigma` $= 2r$ from each
   other, or `np.nan` if they never touch. *Bonus*: write it so that it also works when `dr` and
   `dv` are arrays of shape `(n, 2)`, for $n$ pairs of disks at once. You will need it in Part 4!
   (*Hint*: `np.sum(dr * dv, axis=-1)` computes the dot products, and `np.where` replaces the
   `if` statements.)

2. Write a class `TwoDisks`, very similar to `MovingDisk`, with attributes `positions` and
   `velocities` (arrays of shape `(2, 2)`, one row per disk), a single event (the collision of
   the two disks), and methods `next_event()` and `step(dt)`.

### Check your code

Take two disks of radius $r = 0.2$ with

\begin{equation}
  \vec r_1 = \begin{pmatrix} 0 \\ 0 \end{pmatrix} \qquad
  \vec v_1 = \begin{pmatrix} 1 \\ 0.5 \end{pmatrix} \qquad
  \vec r_2 = \begin{pmatrix} 3 \\ 0 \end{pmatrix} \qquad
  \vec v_2 = \begin{pmatrix} -0.4 \\ 0.6 \end{pmatrix}
\end{equation}

After a time $t = 2.5$, you should find

\begin{equation}
  \vec r_1 = \begin{pmatrix} 1.863 \\ 0.908 \end{pmatrix} \qquad
  \vec v_1 = \begin{pmatrix} -0.045 \\ -0.061 \end{pmatrix} \qquad
  \vec r_2 = \begin{pmatrix} 2.637 \\ 1.842 \end{pmatrix} \qquad
  \vec v_2 = \begin{pmatrix} 0.645 \\ 1.161 \end{pmatrix}
\end{equation}

Check that the total momentum and the total energy are conserved. Watch the collision with
`animate_disks(disks, dt=0.02, n_frames=150, box=None, limits=((-0.5, 4), (-0.5, 3)))`.

### Solution

In the formula for $t_\mathrm{coll}$, the two terms $-b > 0$ and $-\sqrt{b^2 - ac}$ nearly cancel
when the disks are far apart, or when they barely touch: we would lose some digits. Multiplying
the numerator and the denominator by $-b + \sqrt{b^2 - ac}$ gives the equivalent expression
$t_\mathrm{coll} = c / (-b + \sqrt{b^2 - ac})$, where two positive numbers are added.
Because of round-off errors, two disks that just collided can overlap very slightly, $c < 0$, which
would give a tiny negative time: we replace it by zero. Thanks to `axis=-1` and `np.where`, the
function works for a single pair as well as for many pairs at once.

In [ ]:
def collision_time(dr: np.ndarray, dv: np.ndarray, sigma: float) -> np.ndarray | float:
    """Time until two disks at relative position dr, with relative velocity dv, are at a distance sigma.

    dr and dv can have shape (2,) for one pair of disks, or (n, 2) for n pairs at once.
    Impossible collisions are nan.
    """
    a = np.sum(dv * dv, axis=-1)
    b = np.sum(dr * dv, axis=-1)
    c = np.sum(dr * dr, axis=-1) - sigma**2
    discriminant = b**2 - a * c
    with np.errstate(divide="ignore", invalid="ignore"):
        t = c / (-b + np.sqrt(discriminant))  # = (-b - sqrt(discriminant)) / a
    return np.where((b < 0) & (discriminant >= 0), np.maximum(t, 0.0), np.nan)


class TwoDisks:
    """Two disks of the same radius and mass, without walls."""

    def __init__(self, radius: float, positions: np.ndarray, velocities: np.ndarray) -> None:
        self.radius = radius
        self.positions = np.array(positions, dtype=float)
        self.velocities = np.array(velocities, dtype=float)
        self.update_event()

    def update_event(self) -> None:
        """The only event: the collision of the two disks."""
        dr = self.positions[1] - self.positions[0]
        dv = self.velocities[1] - self.velocities[0]
        self.event = collision_time(dr, dv, 2 * self.radius)

    def next_event(self) -> float:
        """Move the disks to the collision, change their velocities and return the elapsed time."""
        t_coll = self.event
        self.positions += t_coll * self.velocities
        r_hat = (self.positions[1] - self.positions[0]) / (2 * self.radius)
        dw = -np.dot(self.velocities[1] - self.velocities[0], r_hat) * r_hat
        self.velocities[0] -= dw
        self.velocities[1] += dw
        self.update_event()
        return t_coll

    def step(self, dt: float) -> None:
        """Evolve the disks during a time dt. A comparison with nan is False: no collision ahead."""
        while self.event <= dt:
            dt -= self.next_event()
        self.positions += dt * self.velocities
        self.event -= dt


disks = TwoDisks(radius=0.2, positions=[[0, 0], [3, 0]], velocities=[[1, 0.5], [-0.4, 0.6]])
momentum, energy = disks.velocities.sum(axis=0), 0.5 * np.sum(disks.velocities**2)
print(f"time to the collision: {disks.event:.4f}")
disks.step(2.5)
np.set_printoptions(precision=3)
print(f"r1 = {disks.positions[0]}, v1 = {disks.velocities[0]}")
print(f"r2 = {disks.positions[1]}, v2 = {disks.velocities[1]}")
print(f"change of momentum: {disks.velocities.sum(axis=0) - momentum}")
print(f"change of energy:   {0.5 * np.sum(disks.velocities**2) - energy:.1e}")

The positions and velocities agree with the expected values, and the momentum and energy are
conserved up to round-off errors. After the collision, the disks never meet again, so the event
becomes `nan`. Since any comparison with `nan` is `False`, the `while` loop of `step` then
simply stops: no special case is needed. Let's watch the collision:

In [ ]:
disks = TwoDisks(radius=0.2, positions=[[0, 0], [3, 0]], velocities=[[1, 0.5], [-0.4, 0.6]])
animate_disks(disks, dt=0.02, n_frames=150, box=None, limits=((-0.5, 4), (-0.5, 3)))

Disk 1 almost stops, while disk 2 leaves with nearly all the momentum: it is not a head-on
collision, but it is not far from it. For a head-on collision, $\Delta\vec v \parallel \hat r$, the
two disks simply exchange their velocities, as in a Newton's cradle.

## Part 4: Many disks

You now have all the ingredients to simulate many disks in a box. To keep things simple, all the
disks have the same radius and mass, and the box is the unit square $[0, 1]^2$.

The events table is now a two-dimensional array of shape `(n_disks, n_disks + 4)`: one row per
disk, one column per possible partner plus walls. The first `n_disks` columns are the collisions with the
other disks, and the last four columns the collisions with the walls (right, top, left, bottom).
The element `events[i, j]` is the time until disk `i` hits disk `j` (or wall `j - n_disks`).

```python
class Disks:
    """Hard disks of the same radius and mass in the unit box."""

    def __init__(self, positions: np.ndarray, velocities: np.ndarray, radius: float) -> None:
        ...
        self.events = np.full((self.n_disks, self.n_disks + 4), np.nan)
        for i in range(self.n_disks):
            self.update_events(i)

    def update_events(self, i: int) -> None:
        """Recompute all the events that involve disk i."""
        ...

    def next_event(self) -> float:
        """Move all the disks to the next event, change the velocities and return the elapsed time."""
        ...

    def step(self, dt: float) -> None:
        """Evolve the disks during a time dt."""
        ...
```

- `update_events(i)` recomputes **row and column** `i` of the table: the times of the
  collisions of disk `i` with all the other disks (the table is symmetric, `events[i, j] =
  events[j, i]`) and with the four walls. A disk does not collide with itself: `events[i, i]` is
  `nan`. With the vectorized `collision_time` of Part 3, the times to all the other disks are
  computed in one line, with `dr = self.positions - self.positions[i]`.

- `next_event()` finds the smallest element of the table. On a 2D array, `np.nanargmin` returns
  an index in the *flattened* array, which `np.unravel_index` turns back into a pair of indices:
  ```python
  i, j = np.unravel_index(np.nanargmin(self.events), self.events.shape)
  ```
  All the disks then move during `t_coll` and all the times in the table decrease by `t_coll`.
  Only the velocities of the disks involved in the event change, so only their rows and columns
  need to be recomputed: one disk for a wall collision, two disks for a disk collision.

- `step(dt)` is the same as before.

### Check your code

Start with two disks and a large radius, and compare with your `TwoDisks` class until they hit a
wall. Then create $20$ disks of radius $0.04$ at random positions, with the function
`random_positions` defined at the top of the notebook, and random velocities, e.g.
`rng.uniform(-1, 1, size=(20, 2))`. Animate them! Do the disks ever overlap? Is the energy
conserved?

### Solution

We reuse the vectorized `collision_time` of Part 3 for the disks, and the same formulas as in
`SingleDisk` for the walls. We also count the number of events and of disk-disk collisions, and
keep track of the time: this will be useful in Part 5.

In [ ]:
class Disks:
    """Hard disks of the same radius and mass in the unit box."""

    def __init__(self, positions: np.ndarray, velocities: np.ndarray, radius: float) -> None:
        self.positions = np.array(positions, dtype=float)
        self.velocities = np.array(velocities, dtype=float)
        self.radius = radius
        self.n_disks = len(self.positions)
        self.time, self.n_events, self.n_collisions = 0.0, 0, 0
        self.events = np.full((self.n_disks, self.n_disks + 4), np.nan)
        for i in range(self.n_disks):
            self.update_events(i)

    def update_events(self, i: int) -> None:
        """Recompute all the events that involve disk i: row and column i of the table."""
        # collisions with all the other disks, at once
        times = collision_time(self.positions - self.positions[i],
                               self.velocities - self.velocities[i], 2 * self.radius)
        times[i] = np.nan  # no collision with itself
        self.events[i, :self.n_disks] = times
        self.events[:, i] = times
        # collisions with the right, top, left and bottom walls
        position, velocity = self.positions[i], self.velocities[i]
        with np.errstate(divide="ignore", invalid="ignore"):
            walls = np.concatenate([(1 - self.radius - position) / velocity, (self.radius - position) / velocity])
        moving_towards = np.concatenate([velocity > 0, velocity < 0])
        self.events[i, self.n_disks:] = np.where(moving_towards, walls, np.nan)

    def next_event(self) -> float:
        """Move all the disks to the next event, change the velocities and return the elapsed time."""
        i, j = np.unravel_index(np.nanargmin(self.events), self.events.shape)
        t_coll = self.events[i, j]
        self.positions += t_coll * self.velocities
        self.events -= t_coll
        if j >= self.n_disks:  # disk i hits a wall
            self.velocities[i, (j - self.n_disks) % 2] *= -1
            self.update_events(i)
        else:                  # disks i and j collide
            r_hat = (self.positions[j] - self.positions[i]) / (2 * self.radius)
            dw = -np.dot(self.velocities[j] - self.velocities[i], r_hat) * r_hat
            self.velocities[i] -= dw
            self.velocities[j] += dw
            self.update_events(i)
            self.update_events(j)
            self.n_collisions += 1
        self.time += t_coll
        self.n_events += 1
        return t_coll

    def step(self, dt: float) -> None:
        """Evolve the disks during a time dt."""
        while np.nanmin(self.events) <= dt:
            dt -= self.next_event()
        self.positions += dt * self.velocities
        self.events -= dt
        self.time += dt

    def energy(self) -> float:
        return 0.5 * np.sum(self.velocities**2)

    def min_distance(self) -> float:
        """The smallest distance between two disks, to check that they never overlap."""
        distances = np.linalg.norm(self.positions[:, None] - self.positions[None, :], axis=-1)
        return np.min(distances + np.diag(np.full(self.n_disks, np.inf)))


# first check: two disks, compared with TwoDisks, before they hit a wall
check = Disks(positions=[[0.3, 0.4], [0.7, 0.4]], velocities=[[0.1, 0.05], [-0.04, 0.06]], radius=0.05)
reference = TwoDisks(radius=0.05, positions=[[0.3, 0.4], [0.7, 0.4]], velocities=[[0.1, 0.05], [-0.04, 0.06]])
check.step(3.0)
reference.step(3.0)
print(f"difference with TwoDisks: {np.max(np.abs(check.positions - reference.positions)):.1e}")

Now $20$ disks at random positions:

In [ ]:
disks = Disks(random_positions(20, 0.04, rng), rng.uniform(-1, 1, size=(20, 2)), radius=0.04)
energy = disks.energy()
display(animate_disks(disks, dt=0.05, n_frames=300))
print(f"{disks.n_events} events, {disks.n_collisions} disk-disk collisions")
print(f"relative change of the energy: {disks.energy() / energy - 1:.1e}")
print(f"smallest distance between two disks: {disks.min_distance():.6f} (diameter: {2 * disks.radius})")

The energy is conserved to machine precision, and the disks never overlap. How fast is our code?

In [ ]:
disks = Disks(random_positions(100, 0.01, rng), rng.uniform(-1, 1, size=(100, 2)), radius=0.01)
%timeit -n 2000 disks.next_event()

About a hundred microseconds per event for $100$ disks. The cost of an event grows like $N^2$,
because `np.nanargmin` scans the whole table, while only $O(N)$ entries change at each event. Real
event-driven codes keep the events in a *priority queue* (the module `heapq` in Python), where the
next event is found in $O(\log N)$ operations, and they only consider the disks that are close to
each other (*cell lists*). This is how millions of disks can be simulated.

## Part 5: From Newton to Boltzmann

Our disks obey Newton's laws, nothing else. Statistical mechanics, on the other hand, predicts
that in equilibrium the velocities are distributed according to the Maxwell-Boltzmann
distribution

\begin{equation}
  \rho(\vec v) = \frac{m}{2\pi k_B T}\, e^{-m v^2 / 2 k_B T}
\end{equation}

in two dimensions. Each component, $v_x$ or $v_y$, is a Gaussian of variance $k_B T / m$, and the
speed $v = |\vec v|$ is distributed according to

\begin{equation}
  \rho(v) = \frac{m v}{k_B T}\, e^{-m v^2 / 2 k_B T}
\end{equation}

But what is the temperature of our gas? It is fixed by the total energy, which is conserved:
with two degrees of freedom per disk, equipartition gives

\begin{equation}
  E = \sum_{i=1}^N \frac{1}{2} m v_i^2 = N k_B T
\end{equation}

Let's put this to the test with a very non-thermal initial state: all the disks have the
**same speed** $|\vec v| = 1$, in random directions.

1. Create $100$ disks at random positions, with unit speeds in random directions. What is
   $k_B T$?
2. Run a long simulation and compute the histogram of the speeds, compared with $\rho(v)$.
   With only $100$ disks, a single
   snapshot gives a noisy histogram: accumulate the speeds of the successive frames. Does the
   histogram converge to the Maxwell-Boltzmann distribution?
4. In another simulation, plot the histogram of the speeds after $1$, $2$ and $20$ collisions per disk on average, and
   compare with $\rho(v)$. This time, average over several independent runs.
5. Do the same for the histogram of $v_x$.

*Hints*: `ax.hist(speeds, bins=40, density=True)` plots a normalized histogram, and
`np.histogram(speeds, bins=bins, density=True)` only computes it. In an animation, the bars drawn by
`bars = ax.bar(...)` can be updated with `bar.set_height(height)`.

### Going further (optional)

- **The billiard break**: put the disks at rest on a square lattice and throw a single fast disk
  at them. Animate! How long does it take for the Maxwell-Boltzmann distribution to set in?
- **The pressure**: every time a disk bounces on a wall, it transfers a momentum $2 m |v_\perp|$ to
  the wall. Measure the pressure $P$, the momentum transferred per unit time and per unit length
  of wall, and compare with the ideal gas law $P A = N k_B T$. What happens at high density?

### Solution

With unit speeds and $m = 1$, the energy is $E = N/2$, so $k_B T = E / N = 1/2$. Each disk-disk
collision involves two disks, so $c$ collisions per disk correspond to $c N / 2$ collisions in total.

#### Watching the gas thermalize

Let's first follow a single, long simulation of $100$ disks. On the left, the disks are colored by
their speed; on the right, the histogram of the speeds. At the beginning all the speeds are equal,
and the histogram is a single peak at $v = 1$. During the first $2$ collisions per disk, we show the
histogram of the current speeds, in slow motion, to see the relaxation. A snapshot of $100$ disks
is noisy, so after that we accumulate the speeds of all the following frames: as the number of
collisions grows, the histogram slowly converges to the Maxwell-Boltzmann distribution.

In [ ]:
def random_unit_velocities(n_disks: int, rng: np.random.Generator) -> np.ndarray:
    angles = rng.uniform(0, 2 * np.pi, size=n_disks)
    return np.column_stack([np.cos(angles), np.sin(angles)])


n_disks, radius = 100, 0.02
gas = Disks(random_positions(n_disks, radius, rng), random_unit_velocities(n_disks, rng), radius)
kT = gas.energy() / n_disks
print(f"k_B T = E / N = {kT:.6f}")
relaxation = 4   # collisions per disk before we start accumulating the speeds
accumulated = []

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4.2), dpi=72, layout="constrained", width_ratios=[1, 1.1])
disks_artist = draw_disks(gas, ax1)
disks_artist.set_array(np.linalg.norm(gas.velocities, axis=1))
disks_artist.set(cmap="Blues", clim=(0, 2.5))
v = np.linspace(0, 3, 300)
bins = np.linspace(0, 3, 31)
bars = ax2.bar(bins[:-1], np.zeros(len(bins) - 1), width=np.diff(bins), align="edge", color=PALETTE[4], alpha=0.7)
ax2.plot(v, v / kT * np.exp(-v**2 / (2 * kT)), ls="dashed", color=PALETTE[1], label="Maxwell-Boltzmann")
ax2.set(xlabel="speed $v$", ylabel=r"$\rho(v)$", xlim=(0, 3), ylim=(0, 1.6))
ax2.legend(loc="upper right", fontsize=10)
title = ax2.set_title("", fontsize=12)
plt.close(fig)

def make_frame(frame: int):
    collisions = 2 * gas.n_collisions / n_disks
    if frame > 0:  # the first frame shows the initial condition
        gas.step(0.004 if collisions < relaxation else 0.02)  # slow motion during the relaxation
    speeds = np.linalg.norm(gas.velocities, axis=1)
    collisions = 2 * gas.n_collisions / n_disks
    if collisions < relaxation:
        sample, label = speeds, "current speeds"
    else:
        accumulated.append(speeds)
        sample, label = np.concatenate(accumulated), f"{len(accumulated)} frames accumulated"
    hist, _ = np.histogram(sample, bins=bins, density=True)
    for bar, height in zip(bars, hist):
        bar.set_height(height)
    disks_artist.set_offsets(gas.positions)
    disks_artist.set_array(speeds)
    title.set_text(f"{collisions:.1f} collisions per disk, {label}")
    return [disks_artist, *bars, title]

# rendering the animation takes about a minute
ani = animation.FuncAnimation(fig, make_frame, frames=500, interval=40, blit=True)
HTML(ani.to_jshtml())

The single peak at $v = 1$ melts away within a couple of collisions per disk, and the histogram of
the accumulated speeds then settles onto the Maxwell-Boltzmann curve. The convergence is slow: two
successive frames are strongly correlated, since only a few disks collide between them. To compare
the distributions at a given number of collisions per disk, it is better to average over
independent runs.

#### After $1$, $2$ and $20$ collisions per disk

We run $50$ independent simulations, and take a snapshot of the velocities each time the number of
collisions per disk reaches $1$, $2$ and $20$. Here we use smaller disks, of radius $0.01$.

In [ ]:
n_disks, radius, n_runs = 100, 0.01, 50
collisions_per_disk = [1, 2, 20]
snapshots = {c: [] for c in collisions_per_disk}
for _ in range(n_runs):
    disks = Disks(random_positions(n_disks, radius, rng), random_unit_velocities(n_disks, rng), radius)
    for c in collisions_per_disk:
        while disks.n_collisions < c * n_disks / 2:
            disks.next_event()
        snapshots[c].append(disks.velocities.copy())
kT = disks.energy() / n_disks
print(f"k_B T = E / N = {kT:.6f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4), layout="constrained")
v = np.linspace(0, 3, 300)
colors = plt.cm.Blues(np.linspace(0.45, 1, len(collisions_per_disk)))
ax1.axvline(1, color="#898781", lw=1.5, ls=":", label="initial speed")
for (c, velocities), color in zip(snapshots.items(), colors):
    speeds = np.linalg.norm(np.concatenate(velocities), axis=1)
    ax1.hist(speeds, bins=np.linspace(0, 3, 41), density=True, histtype="step", lw=2, color=color,
             label=f"{c} collision{'s' if c > 1 else ''} per disk")
ax1.plot(v, v / kT * np.exp(-v**2 / (2 * kT)), ls="dashed", color=PALETTE[1], label="Maxwell-Boltzmann")
ax1.set(xlabel="speed $v$", ylabel=r"$\rho(v)$", xlim=(0, 3), ylim=(0, 1.6), title="Distribution of the speeds")
ax1.legend(fontsize=10)

vx = np.concatenate(snapshots[20])[:, 0]
u = np.linspace(-2.5, 2.5, 300)
ax2.hist(vx, bins=np.linspace(-2.5, 2.5, 41), density=True, color=PALETTE[4], alpha=0.6, label="20 collisions per disk")
ax2.plot(u, np.exp(-u**2 / (2 * kT)) / np.sqrt(2 * np.pi * kT), ls="dashed", color=PALETTE[1], label="Gaussian")
ax2.set(xlabel="$v_x$", ylabel=r"$\rho(v_x)$", xlim=(-2.5, 2.5), title="Distribution of a velocity component")
ax2.legend(fontsize=10);

All the disks start with $v = 1$. After a single collision per disk, the distribution has already
spread over all speeds, and after two collisions it is not far from the Maxwell-Boltzmann
distribution. After $20$ collisions per disk, the agreement is perfect, for the speeds as well as
for the velocity components. The temperature that appears in the distribution is exactly the one
fixed by the energy, $k_B T = 1/2$.

This is a beautiful result: the motion of the disks is deterministic and reversible, and yet the
gas relaxes irreversibly to the equilibrium predicted by statistical mechanics. The collisions
scramble the velocities so efficiently that, after a few of them, every disk seems to have
forgotten its initial velocity: this is Boltzmann's *molecular chaos*.

Is this a coincidence of our simulation? No: it is a theorem, at least for closely related systems.
In 1970, Sinai proved that a system of two hard disks on a torus is *ergodic* and even *mixing*:
almost every trajectory explores the whole energy surface uniformly, so that time averages coincide
with the averages of the microcanonical ensemble, and a non-equilibrium initial state relaxes to
equilibrium (Y. G. Sinai, *Dynamical systems with elastic reflections*, Russian Mathematical Surveys
**25**, 137 (1970)). Extending the proof to $N$ hard disks or balls, the *Boltzmann-Sinai ergodic
hypothesis*, took several more decades of work. Strictly speaking, ergodicity predicts the
microcanonical distribution of the speed of one disk among $N$, which only becomes the
Maxwell-Boltzmann distribution for $N \to \infty$; for $100$ disks the difference is far too small
to be seen on our histograms.

### Going further: the billiard break

$100$ disks at rest on a square lattice, and a fast disk thrown at them. The disks at rest never
collide with each other ($\Delta\vec v = 0$ gives `nan`), so the code needs no change. The color
of each disk shows its speed.

In [ ]:
grid = (np.arange(10) + 0.5) / 10
positions = np.array([(x, y) for x in grid for y in grid])
positions[0] = (0.02, 0.07)              # the projectile, in the corner...
velocities = np.zeros_like(positions)
velocities[0] = (10.0, 7.0)              # ...thrown at the others
break_shot = Disks(positions, velocities, radius=0.015)
energy = break_shot.energy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4.2), dpi=72, layout="constrained", width_ratios=[1, 1.1])
disks_artist = draw_disks(break_shot, ax1)
disks_artist.set_array(np.linalg.norm(break_shot.velocities, axis=1))
disks_artist.set(cmap="Blues", clim=(0, 3 * np.sqrt(energy / len(positions))))
v = np.linspace(0, 4, 300)
kT = energy / len(positions)
ax2.plot(v, v / kT * np.exp(-v**2 / (2 * kT)), ls="dashed", color=PALETTE[1], label="Maxwell-Boltzmann")
bins = np.linspace(0, 4, 31)
bars = ax2.bar(bins[:-1], np.zeros(len(bins) - 1), width=np.diff(bins), align="edge", color=PALETTE[4], alpha=0.7)
ax2.set(xlabel="speed $v$", ylabel=r"$\rho(v)$", xlim=(0, 4), ylim=(0, 1.6))
ax2.legend(loc="upper right", fontsize=10)
title = ax2.set_title("")
plt.close(fig)

def make_frame(frame: int):
    if frame > 0:  # the first frame shows the initial condition
        break_shot.step(0.005)
    speeds = np.linalg.norm(break_shot.velocities, axis=1)
    disks_artist.set_offsets(break_shot.positions)
    disks_artist.set_array(speeds)
    hist, _ = np.histogram(speeds, bins=bins, density=True)
    for bar, height in zip(bars, hist):
        bar.set_height(height)
    title.set_text(f"{2 * break_shot.n_collisions / len(positions):.1f} collisions per disk")
    return [disks_artist, *bars, title]

ani = animation.FuncAnimation(fig, make_frame, frames=500, interval=40, blit=True)
HTML(ani.to_jshtml())

A cascade of collisions spreads the energy of the projectile through the whole gas. After a few
collisions per disk, the histogram of the speeds already fluctuates around the Maxwell-Boltzmann
distribution (with only $100$ disks, a single snapshot is noisy).

### Going further: the pressure

We count the momentum $2 |v_\perp|$ given to the walls during a long time $T$, after a short
equilibration. The pressure is this momentum divided by $T$ and by the perimeter $4$ of the box.
We start from a square lattice, which allows high densities, and vary the packing fraction
$\eta = N \pi r^2$, the fraction of the box covered by the disks. We compare with the ideal gas,
$P / (N k_B T) = 1$ for a box of area $1$, and with the empirical equation of state of
hard disks by Henderson (1975), $P A / (N k_B T) = (1 + \eta^2/8) / (1 - \eta)^2$.

In [ ]:
class DisksWithPressure(Disks):
    """Hard disks that record the momentum transferred to the walls."""

    def __init__(self, positions: np.ndarray, velocities: np.ndarray, radius: float) -> None:
        super().__init__(positions, velocities, radius)
        self.wall_momentum = 0.0

    def next_event(self) -> float:
        i, j = np.unravel_index(np.nanargmin(self.events), self.events.shape)
        if j >= self.n_disks:
            self.wall_momentum += 2 * abs(self.velocities[i, (j - self.n_disks) % 2])
        return super().next_event()


n_side = 10
n_disks = n_side**2
grid = (np.arange(n_side) + 0.5) / n_side
lattice = np.array([(x, y) for x in grid for y in grid])
packing_fractions = np.array([0.02, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
compressibility = []
for eta in packing_fractions:
    radius = np.sqrt(eta / (n_disks * np.pi))
    gas = DisksWithPressure(lattice, random_unit_velocities(n_disks, rng), radius)
    gas.step(2.0)                                 # equilibration
    gas.wall_momentum, start = 0.0, gas.time
    gas.step(10.0)
    pressure = gas.wall_momentum / ((gas.time - start) * 4)
    compressibility.append(pressure / gas.energy())  # P A / (N k_B T), with A = 1 and N k_B T = E

eta = np.linspace(0, 0.65, 200)
fig, ax = plt.subplots()
ax.axhline(1, color="#898781", lw=1.5, ls=":", label="ideal gas")
ax.plot(eta, (1 + eta**2 / 8) / (1 - eta) ** 2, ls="dashed", color=PALETTE[1], label="Henderson")
ax.plot(packing_fractions, compressibility, "o", ms=8, color=PALETTE[0], label=f"simulation, $N = {n_disks}$")
ax.set(xlabel=r"packing fraction $\eta$", ylabel=r"$P A / N k_B T$", title="Equation of state of hard disks")
ax.legend();

At low density, the gas is ideal. As the density increases, the pressure grows much faster than the
ideal gas law predicts: each disk has less free space to move in, and it hits the others, and the
walls, more often. This is the *excluded volume* effect, the same that appears in the van der
Waals equation. The simulation follows Henderson's equation of state, within about ten percent at
the highest densities. The differences come from the walls: in a box of only $100$ disks, a large fraction of the disks are
close to a wall, where the gas is not the same as in the bulk. Near $\eta \simeq 0.7$, hard disks
melt into a solid: Alder and Wainwright saw the first signs of this transition in 1962, and it took
until 2011, with huge event-driven simulations of a million disks, to understand it completely.